# WSI cleanup acceptance (user-run only)
Read ../WSI_POST_RESOLUTION_CLEANUP.md first. This notebook does not submit jobs. Use an existing allocation and a separate, reviewed acceptance YAML with new output paths. Run the representative half crop before a full slide. Keep RUN_INFERENCE=False until inputs and output isolation are reviewed. Preserve pre-cleanup diagnostic labels separately when collecting removal panels; normal production completion deletes work-Zarr. Record peak RSS from the job accounting and cleanup/validation wall time from the run log.

In [ ]:
from pathlib import Path
from mif_pipeline import load_config, run_instanseg
RUN_INFERENCE = False
CONFIG_PATH = Path("/replace/with/reviewed/acceptance.yaml")
SLIDE_ID = "SLIDE-0330"


In [ ]:
if RUN_INFERENCE:
    from mif_pipeline.config import get_slide_config
    config = load_config(CONFIG_PATH)
    slide = get_slide_config(config, SLIDE_ID)
    expected = {"mode": "wsi_global", "resolution_method": "watershed",
                "resolve_cell_and_nucleus": True, "seed_threshold": 0.2,
                "cleanup_fragments": True, "min_size": 10,
                "cleanup_resolved_fragments": True, "allow_unnucleated_cells": False}
    assert all(slide["instanseg"].get(k) == v for k, v in expected.items())
    print(slide["full_merge"]["ome_path"], slide["mask_export"]["mask_dir"])
    result = run_instanseg(config, SLIDE_ID)  # Never force existing outputs here.
    print(result)
else:
    print("Inference disabled. Prepare and review isolated acceptance paths first.")


Do not mark acceptance complete from a valid manifest alone. Follow the visual review, removal-rate, runtime, memory, restart, Nimbus and SpatialData gates in the durable document. Seed 0.2 remains provisional until those checks pass.